In [6]:
#Read in data with TEST fingerprints included
import pandas as pd
data = pd.read_csv('../src/data/LD50_pre-grid-search_and_TEST.csv').set_index("Unnamed: 0")

In [8]:
from sklearn.utils.validation import check_array
import numpy as np

#Import things that will be affected by these functions
from sklearn.model_selection import train_test_split
from genra.rax.skl.hybrid import GenRAPredValueHybrid
from numpy import sqrt
from sklearn.metrics import r2_score

#GenRA currrently does not support single-sample predictions for metrics other than the binary Jaccard metric,
# so this code will circumvent that with small alterations to the source code. 
def kneighbors_sim(self,X):
    """
    Find the k-nearest neighbours for each instance and similarity scores. 
    All distances (D) are converted to similarity (S) by:
    
                D - D.min()
    Sim =   --------------
            D.max()-D.min()
    We assume D.min()==0

    """
    neigh_dist, neigh_ind = self.kneighbors(X)
    
    """If the metric used has a max of 1, use this commented-out version of the following code. For metrics
    with larger possible metrics (esp. canberra), you will need to restore this to the edited form without the 
    comment blocks"""
    
    # Convert distances to similarities:
    # if self.metric == 'jaccard':
    neigh_sim = 1-neigh_dist
    # else:
    #     if neigh_dist.max() > 0:
    #         neigh_dist_n = neigh_dist / neigh_dist.max()
    #         neigh_sim = 1 - neigh_dist_n
    #     else:
    #         neigh_sim = 1
                    
    
    return neigh_sim, neigh_ind

def predict(self, X):
    """Predict the target for the provided data

    Parameters
    ----------
    X : array-like, shape (n_queries, n_features), \
            or (n_queries, n_indexed) if metric == 'precomputed'
        Test samples.

    Returns
    -------
    y : array of int, shape = [n_queries] or [n_queries, n_outputs]
        Target values
    """
    X = check_array(X, accept_sparse='csr')

    neigh_sim, neigh_ind = kneighbors_sim(self,X)
    
    _y = self._y
    if _y.ndim == 1:
        _y = _y.reshape((-1, 1))

    y_pred = np.empty((X.shape[0], _y.shape[1]), dtype=np.float64)

    denom=np.sum(neigh_sim)

    for j in range(_y.shape[1]):
        num = np.sum(_y[neigh_ind, j] * neigh_sim, axis=1)
        if denom > 0:
            y_pred[:, j] = num / denom
        else:
            denom = len(neigh_ind)
            y_pred[:, j] = num / denom
            
    if self._y.ndim == 1:
        y_pred = y_pred.ravel()

    return y_pred

In [35]:
#Here we create a generalized Jaccard metric, which has been tested for consistency with the pre-built
#Jaccard metric on binary sets

def generalJaccard(row1, row2):
    diff = np.array(row1)-np.array(row2)
    denom = (np.dot(diff, diff)+np.dot(row1, row2))
    if denom != 0:
        similarity = np.dot(row1, row2)/(np.dot(diff, diff)+np.dot(row1, row2))
    else:
        similarity = 0
    return similarity

def generalJaccardDistance(row1, row2):
    similarity = generalJaccard(row1, row2)
    distance = 1 - similarity
    return distance

In [42]:
#Test the performance of the TEST fingerprints alone or Morgan, for comparison
scores = {"state":[], "r2":[], 'rmse':[]}
for state in [4745,134,907, 654, 607, 90, 9054,9,12]:
    slices = [slice(0,729),slice(729,2777), slice(2777,4825), slice(4825, 5727), slice(5727, None)]
    x_train, x_test, y_train, y_test = train_test_split(data.iloc[:,1:], data.iloc[:,0], random_state=state)
    tester = GenRAPredValueHybrid(n_neighbors =8, slices = slices, hybrid_weights=[0,100,0,0,0],metric = 'cosine')
    y_preds = []

    for x in x_test.iterrows():
        tester.fit(data.loc[data.index != x[0]].iloc[:, 1:], data.loc[data.index != x[0]].iloc[:,0])
        y_preds.append(predict(tester,list(x[1:])))
    r2 = r2_score(y_test, y_preds)
    rmse = sqrt((sum([(y_preds[i]-y_test.values[i])**2 for i in range(0,len(y_preds))])/len(y_preds)))
    scores['r2'].append(r2)
    scores['rmse'].append(rmse)
    scores['state'].append(state)
    scores_df = pd.DataFrame(scores)
    scores_df.to_csv('metric_test_TEST_cosine_Morgan.csv')

In [16]:
slices = [slice(0,729),slice(729,2777), slice(2777,4825), slice(4825, 5727), slice(5727, None)]
tester = GenRAPredValueHybrid(n_neighbors =8, slices = slices, hybrid_weights=[0,100,0,0,0],metric = 'cosine')

tester.fit(data.iloc[5:, 1:], data.iloc[5:,0])
print(f"The prediction for {data.index[0]} is {tester.predict(data.iloc[0:1,1:])[0]} when it is the only member of the test set.")

The prediction for DTXSID5020281 is -1.0596185268821183 when it is the only member of the test set.


In [18]:
print(f"The prediction for {data.index[0]} is {tester.predict(data.iloc[0:5, 1:])[0]} if we include \n {list(data.index[0:5])} in the test set.")

The prediction for DTXSID5020281 is -0.8684254005846935 if we include 
 ['DTXSID5020281', 'DTXSID8020961', 'DTXSID0021834', 'DTXSID2044347', 'DTXSID4025745'] in the test set.


In [19]:
print(f"The actual LD50_LM value for {data.index[0]} is {data.iloc[0,0]} ")

The actual LD50_LM value for DTXSID5020281 is -0.465339415759 


In [6]:
#Import the neighborhood results
print('scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test_TEST_canberra10_TEST.csv references/TEST_tests/')
print('scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test_TEST_jaccard_morgan.csv references/TEST_tests/')

scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test_TEST_canberra10_TEST.csv references/TEST_tests/
scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test_TEST_jaccard_morgan.csv references/TEST_tests/


In [23]:
print('scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test2_canberra13_TEST.csv references/TEST_tests/')
print('scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test2_jaccard_morgan.csv references/TEST_tests/')
print('scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test2_jaccard_opt.csv references/TEST_tests/')

scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test2_canberra13_TEST.csv references/TEST_tests/
scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test2_jaccard_morgan.csv references/TEST_tests/
scp aleary@v2626umcth031.rtord.epa.gov:/home/aleary/Grid-Search_Optimization/outputs/LD50/TEST/full_test2_jaccard_opt.csv references/TEST_tests/


In [30]:
import pandas as pd

TEST_df = pd.read_csv("../references/TEST_tests/full_test_TEST_canberra10_TEST.csv").set_index("Unnamed: 0")
morgan_df = pd.read_csv('../references/TEST_tests/full_test_TEST_jaccard_morgan.csv').set_index("Unnamed: 0")


reTEST_df = pd.read_csv("../references/TEST_tests/full_test2_canberra13_TEST.csv").set_index("Unnamed: 0")
remorgan_df = pd.read_csv('../references/TEST_tests/full_test2_jaccard_morgan.csv').set_index("Unnamed: 0")
reopt_df = pd.read_csv('../references/TEST_tests/full_test2_jaccard_opt.csv').set_index("Unnamed: 0")

TEST_df['rmse'] = [float(i[1:-1]) for i in TEST_df['rmse']]
morgan_df['rmse'] = [float(i[1:-1]) for i in morgan_df['rmse']]

reTEST_df['rmse'] = [float(i[1:-1]) for i in reTEST_df['rmse']]
remorgan_df['rmse'] = [float(i[1:-1]) for i in remorgan_df['rmse']]
reopt_df['rmse'] = [float(i[1:-1]) for i in reopt_df['rmse']]

TEST_df = TEST_df.rename(columns = {'r2':f'TEST_r2', 'rmse':'TEST_rmse'})
morgan_df = morgan_df.rename(columns = {'r2':f'morgan_r2', 'rmse':'morgan_rmse'})


reTEST_df = reTEST_df.rename(columns = {'r2':f'TEST_r2', 'rmse':'TEST_rmse'})
remorgan_df = remorgan_df.rename(columns = {'r2':f'morgan_r2', 'rmse':'morgan_rmse'})
reopt_df = reopt_df.rename(columns = {'r2':f'optimum_r2', 'rmse':'optimum_rmse'})

metric_comp_df = pd.merge(TEST_df, morgan_df, on = 'state')
new_metric_comp_df = pd.merge(pd.merge(reTEST_df, remorgan_df, on = 'state'), reopt_df, on = 'state')


In [8]:
metric_comp_df.mean(), metric_comp_df.std()

(state          4525.200000
 TEST_r2           0.546959
 TEST_rmse         0.610430
 morgan_r2         0.535013
 morgan_rmse       0.618541
 dtype: float64,
 state          3010.912073
 TEST_r2           0.024051
 TEST_rmse         0.016328
 morgan_r2         0.023464
 morgan_rmse       0.019541
 dtype: float64)

In [32]:
new_metric_comp_df['changeTEST-mor'] = new_metric_comp_df['TEST_r2'] - new_metric_comp_df['morgan_r2']
new_metric_comp_df['changeTEST-opt'] = new_metric_comp_df['TEST_r2'] - new_metric_comp_df['optimum_r2']
new_metric_comp_df[['state', 'TEST_r2', 'morgan_r2', 'optimum_r2', 'changeTEST-mor', 'changeTEST-opt']]

,state,TEST_r2,morgan_r2,optimum_r2,changeTEST-mor,changeTEST-opt
0,6702,0.554321,0.517022,0.546278,0.037298,0.008042
1,8074,0.568376,0.548897,0.588598,0.019479,-0.020222
2,7908,0.557403,0.519575,0.533991,0.037828,0.023412
3,1975,0.545858,0.543116,0.572320,0.002742,-0.026463
4,9517,0.545111,0.535936,0.567087,0.009175,-0.021976
5,3544,0.527664,0.521630,0.538346,0.006034,-0.010683
6,5809,0.532302,0.514625,0.527539,0.017677,0.004763
7,4397,0.575261,0.554071,0.566952,0.021190,0.008308
8,6005,0.555747,0.533964,0.560586,0.021783,-0.004839
9,2010,0.572399,0.558380,0.595623,0.014019,-0.023224


In [10]:
from numpy import sqrt
test_mean_r2 = metric_comp_df.mean()['TEST_r2']
mor_mean_r2 = metric_comp_df.mean()['morgan_r2']
test_sd_r2 = metric_comp_df.std()['TEST_r2']
mor_sd_r2 = metric_comp_df.std()['morgan_r2']
t_statistic =  (test_mean_r2 - mor_mean_r2)/(sqrt((test_sd_r2**2)/10+(mor_sd_r2**2)/10))
t_statistic

1.1242995541255891

In [22]:
from numpy import sqrt
test_mean_r2 = new_metric_comp_df.mean()['TEST_r2']
mor_mean_r2 = new_metric_comp_df.mean()['morgan_r2']
test_sd_r2 = new_metric_comp_df.std()['TEST_r2']
mor_sd_r2 = new_metric_comp_df.std()['morgan_r2']
t_statistic =  (test_mean_r2 - mor_mean_r2)/(sqrt((test_sd_r2**2)/10+(mor_sd_r2**2)/10))
t_statistic

2.606374425794077